# BCNN Edge Vision Accelerator — Phase A

**Target:** Cyclone IV EP4CE115F29C7 · Quartus II 13.0 · 32×32 grayscale input

| Step | Description |
|------|-------------|
| 1 | Dataset — Rock Paper Scissors (TFDS, auto-downloaded) |
| 2 | Teacher — MobileNetV2 (ImageNet weights, fine-tuned on RPS) |
| 3 | Student — Tiny BCNN with **STE binary weights + binary activations** |
| 4 | Knowledge Distillation training (soft labels from teacher) |
| 5 | Post-training weight binarization & BatchNorm folding |
| 6 | Hardware file generation (.mif / .hex for Quartus ROMs) |
| 7 | Software XNOR inference — vectorized NumPy (RTL golden reference) |
| 8 | XNORNet α correction + logistic regression dense-head refit |
| 9 | Inference timing (CPU baseline for FPGA speedup comparison) |
| 10 | Summary & zip export |

> **Why STE instead of post-hoc binarization?**  
> Post-hoc binarization (just calling `sign()` after float training) produces
> weights that were *never optimised to be binary*.  The Straight-Through
> Estimator runs `sign()` in every forward pass during training so gradients
> push weights toward ±1 naturally.  Combined with KD soft labels, the student
> learns binary-friendly features from a much stronger teacher.


## Step 0 — Environment & Constants

In [ ]:
# ── Install optional packages if missing ──────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import tensorflow_datasets
except ImportError:
    pip_install("tensorflow-datasets")

try:
    from tqdm.auto import tqdm
except ImportError:
    pip_install("tqdm")
    from tqdm.auto import tqdm

# ── Core imports ──────────────────────────────────────────────────────────────
import os, json, struct, math, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from scipy.special import softmax as scipy_softmax

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

# ── GPU setup — enable memory growth to avoid OOM on Colab ───────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU memory growth enabled — {len(gpus)} GPU(s) available")
else:
    print("⚠  No GPU detected — training will run on CPU")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Image dimensions (match RTL) ──────────────────────────────────────────────
IMG_H, IMG_W, IMG_C = 32, 32, 1
NUM_CLASSES          = 2          # 0 = Rock, 1 = Not-Rock
CONV1_FILTERS        = 8
CONV2_FILTERS        = 16
KERNEL_SIZE          = 3
PIXEL_BIN_THR        = 128        # input binarization threshold (0–255)

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE           = 64
TEACHER_EPOCHS_P1    = 20         # Phase 1: frozen backbone
TEACHER_EPOCHS_P2    = 10         # Phase 2: unfreeze top 30 layers
STUDENT_EPOCHS       = 60
TEACHER_LR           = 1e-3       # Phase 1 learning rate
STUDENT_LR           = 3e-4       # Student KD learning rate

# ── Knowledge distillation ────────────────────────────────────────────────────
KD_TEMPERATURE       = 4.0        # soften teacher distributions
KD_ALPHA             = 0.7        # fraction of loss from soft labels

# ── Fixed-point spec (Q8.8, used for dense layer in RTL) ─────────────────────
FP_SCALE = 256                    # 2^8
FP_CLIP  = (-32768, 32767)        # int16 range

# ── Output directories ────────────────────────────────────────────────────────
OUTPUT_DIR = Path("output")
MEM_DIR    = OUTPUT_DIR / "mem"
HEX_DIR    = OUTPUT_DIR / "hex"
MODEL_DIR  = OUTPUT_DIR / "models"
PLOT_DIR   = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, MEM_DIR, HEX_DIR, MODEL_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

fp_spec = {"format": "Q8.8", "scale": FP_SCALE, "type": "int16",
           "clipping": list(FP_CLIP)}
with open(OUTPUT_DIR / "fixed_point_spec.json", "w") as f:
    json.dump(fp_spec, f, indent=2)

print(f"\nAll outputs → {OUTPUT_DIR.resolve()}")


## Step 1 — Dataset: Rock Paper Scissors

Binary classification: **Rock = 0, Not-Rock (Paper + Scissors) = 1**.

This maps directly to a live Nicla Vision test: closed fist → class 0,
open hand or scissors → class 1. Immediate visual feedback, no printed props.

Dataset is loaded from `tensorflow_datasets` (~220 MB, downloaded once).


In [ ]:
import tensorflow_datasets as tfds

ROCK_LABEL_TFDS = 1   # TFDS alphabetical order: 0=paper, 1=rock, 2=scissors

def preprocess(img: tf.Tensor, label: tf.Tensor):
    """Resize → grayscale → normalise → remap label (Rock=0, else=1)."""
    img   = tf.image.rgb_to_grayscale(img)
    img   = tf.image.resize(img, [IMG_H, IMG_W])
    img   = tf.cast(img, tf.float32) / 255.0
    label = tf.cast(tf.not_equal(label, ROCK_LABEL_TFDS), tf.int32)
    return img, label

def ds_to_numpy(ds) -> tuple[np.ndarray, np.ndarray]:
    """Batch-extract numpy arrays from a tf.data.Dataset (much faster than per-sample)."""
    ds_batched = (
        ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(256)
          .prefetch(tf.data.AUTOTUNE)
    )
    imgs_list, labels_list = [], []
    for img_batch, lbl_batch in ds_batched:
        imgs_list.append(img_batch.numpy())
        labels_list.append(lbl_batch.numpy())
    return (np.concatenate(imgs_list,   axis=0).astype(np.float32),
            np.concatenate(labels_list, axis=0).astype(np.int32))

print("Loading Rock-Paper-Scissors dataset …")
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    "rock_paper_scissors",
    split=["train", "test"],
    as_supervised=True,
    with_info=True,
    shuffle_files=False,
)

X_tfds_train, y_tfds_train = ds_to_numpy(ds_train_raw)
X_tfds_test,  y_tfds_test  = ds_to_numpy(ds_test_raw)

X_all = np.concatenate([X_tfds_train, X_tfds_test])
y_all = np.concatenate([y_tfds_train, y_tfds_test])

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(y_all))
X_all, y_all = X_all[idx], y_all[idx]

print(f"Total samples : {len(X_all)}")
print(f"Shape         : {X_all.shape}")
print(f"Rock (0) / Not-Rock (1) : {(y_all==0).sum()} / {(y_all==1).sum()}")

# ── Train / Val / Test split (70 / 15 / 15) ──────────────────────────────────
X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=SEED
)

print(f"Train : {len(X_train)}  Val : {len(X_val)}  Test : {len(X_test)}")

# ── Quick visualisation ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle("Top: Rock (0)  |  Bottom: Not-Rock (1)", fontsize=12)
rock_idx    = np.where(y_all == 0)[0][:8]
notrock_idx = np.where(y_all == 1)[0][:8]
for j in range(8):
    axes[0, j].imshow(X_all[rock_idx[j], ..., 0],    cmap="gray", vmin=0, vmax=1)
    axes[0, j].axis("off")
    axes[1, j].imshow(X_all[notrock_idx[j], ..., 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR / "dataset_samples.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 2 — Teacher Model (MobileNetV2, ImageNet → RPS)

The teacher is a full float32 model — we want it as accurate as possible so
it produces rich soft labels for the student.  It is **never deployed to the
FPGA**; it only exists during training.

Training is split into two phases:
1. **Phase 1 (frozen backbone):** train only the classification head (fast)
2. **Phase 2 (partial unfreeze):** unfreeze the top 30 MobileNetV2 layers and
   fine-tune at a low learning rate to squeeze out extra accuracy

The teacher outputs **logits** (no softmax) so we can apply temperature
scaling directly when computing soft labels for KD.


In [ ]:
def build_teacher() -> keras.Model:
    """MobileNetV2 backbone + small classification head, outputs logits."""
    backbone = tf.keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(96, 96, 3),
    )
    backbone.trainable = False   # frozen for Phase 1

    inp = keras.Input(shape=(IMG_H, IMG_W, IMG_C), name="teacher_input")

    # Grayscale → RGB, upsample to MobileNetV2 minimum, preprocess
    x = layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1), name="grey2rgb")(inp)
    x = layers.Resizing(96, 96, name="resize96")(x)
    x = layers.Lambda(
        lambda t: tf.keras.applications.mobilenet_v2.preprocess_input(t * 255.0),
        name="mn_preprocess",
    )(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(NUM_CLASSES, name="logits")(x)   # no softmax — logits

    return keras.Model(inputs=inp, outputs=out, name="MobileNetV2_teacher")


teacher_model = build_teacher()

# ── Phase 1: train head only ──────────────────────────────────────────────────
print("Phase 1 — training classification head (backbone frozen) …")

cw_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw_arr)}
print(f"Class weights: {class_weight}")

teacher_model.compile(
    optimizer=Adam(TEACHER_LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

history_t1 = teacher_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=TEACHER_EPOCHS_P1,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                restore_best_weights=True, verbose=0),
    ],
    verbose=2,
)

y_pred_t1 = np.argmax(teacher_model.predict(X_test, verbose=0), axis=1)
acc_t1 = accuracy_score(y_test, y_pred_t1)
print(f"\nPhase 1 teacher accuracy: {acc_t1*100:.2f}%")


In [ ]:
# ── Phase 2: unfreeze top 30 layers and fine-tune ────────────────────────────
print("Phase 2 — fine-tuning top 30 backbone layers …")

backbone_layer = teacher_model.get_layer("mobilenetv2_1.00_96")
backbone_layer.trainable = True

# Only unfreeze the last 30 layers; keep earlier feature extractors frozen
for layer in backbone_layer.layers[:-30]:
    layer.trainable = False

teacher_model.compile(
    optimizer=Adam(TEACHER_LR / 10),   # lower LR to avoid destroying ImageNet features
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

ckpt_path = str(MODEL_DIR / "teacher_best.keras")
history_t2 = teacher_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=TEACHER_EPOCHS_P2,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy",
                                  save_best_only=True, mode="max", verbose=0),
        callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                restore_best_weights=True, verbose=0),
    ],
    verbose=2,
)

y_pred_teacher = np.argmax(teacher_model.predict(X_test, verbose=0), axis=1)
teacher_acc = accuracy_score(y_test, y_pred_teacher)
print(f"\nFinal teacher accuracy: {teacher_acc*100:.2f}%")
print(classification_report(y_test, y_pred_teacher,
                             target_names=["Rock", "Not-Rock"]))


## Step 3 — Student Model with STE Binary Weights & Activations

### Why STE?

Post-hoc binarization (`sign()` after float training) applies binarization to
weights that were optimised *as floats*.  Many weights sit at small magnitudes
(e.g. `0.07`, `-0.23`) where the sign is essentially arbitrary — the model
never learned to commit to ±1.

The **Straight-Through Estimator** fixes this:

```
Forward pass:  use sign(W_float)  →  binary XNOR arithmetic
Backward pass: treat sign() as identity  →  gradients flow normally
```

After thousands of steps with binarized weights in every forward pass,
gradient descent naturally pushes weights toward ±1.  The final `sign()`
call in Step 5 causes almost zero information loss.

We also binarize **activations** with the same STE trick, so the training
forward pass exactly mirrors RTL behaviour: XNOR-popcount → threshold → ±1.

### Architecture
```
Input (32×32×1)
  ↓  [augmentation — training only]
  ↓  BinaryConv2D(8,  3×3, same)  →  BN  →  BinaryActivation  (±1)
  ↓  BinaryConv2D(16, 3×3, same)  →  BN  →  BinaryActivation  (±1)
  ↓  GlobalAveragePooling2D
  ↓  Dropout(0.3)
  ↓  Dense(2)  →  logits
```


In [ ]:
# ── STE primitives ────────────────────────────────────────────────────────────

@tf.custom_gradient
def ste_sign(x):
    """
    Forward : sign(x), with sign(0) = +1.
    Backward: identity gradient clipped to |x| <= 1  (Straight-Through Estimator).
    """
    y = tf.sign(x)
    y = tf.where(tf.equal(x, 0.0), tf.ones_like(x), y)

    def grad(dy):
        return dy * tf.cast(tf.abs(x) <= 1.0, tf.float32)

    return y, grad


class BinaryConv2D(layers.Layer):
    """
    Conv2D with binary weights via STE.

    The underlying kernel is stored as a full float32 weight (so gradients
    can accumulate).  During the forward pass the kernel is binarized with
    ste_sign() before the convolution, so the network always 'sees' ±1 weights.
    """
    def __init__(self, filters, kernel_size, padding="same",
                 kernel_regularizer=None, **kwargs):
        super().__init__(**kwargs)
        self.filters            = filters
        self.kernel_size        = kernel_size
        self.padding            = padding.upper()
        self.kernel_regularizer = tf.keras.regularizers.get(kernel_regularizer)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(self.kernel_size, self.kernel_size,
                   int(input_shape[-1]), self.filters),
            initializer="glorot_uniform",
            regularizer=self.kernel_regularizer,
            trainable=True,
        )

    def call(self, x):
        W_bin = ste_sign(self.kernel)          # binarize forward, STE backward
        return tf.nn.conv2d(x, W_bin, strides=1, padding=self.padding)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(filters=self.filters, kernel_size=self.kernel_size,
                   padding=self.padding.lower())
        return cfg


class BinaryActivation(layers.Layer):
    """sign() activation with STE gradient — mirrors RTL threshold comparison."""
    def call(self, x):
        return ste_sign(x)


# ── Student model ─────────────────────────────────────────────────────────────

def build_student() -> keras.Model:
    inp = keras.Input(shape=(IMG_H, IMG_W, IMG_C), name="input")

    # Augmentation (disabled at inference time automatically)
    x = layers.RandomFlip("horizontal",       name="aug_flip")(inp)
    x = layers.RandomRotation(0.1,            name="aug_rotate", fill_mode="nearest")(x)
    x = layers.RandomZoom(0.1,                name="aug_zoom",   fill_mode="nearest")(x)
    x = layers.RandomContrast(0.2,            name="aug_contrast")(x)

    # Layer 1: binary conv → BN → binary activation
    x = BinaryConv2D(CONV1_FILTERS, KERNEL_SIZE,
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                     name="bin_conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = BinaryActivation(name="bin_act1")(x)

    # Layer 2: binary conv → BN → binary activation
    x = BinaryConv2D(CONV2_FILTERS, KERNEL_SIZE,
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                     name="bin_conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = BinaryActivation(name="bin_act2")(x)

    # Head: GAP → dropout → dense (logits)
    x   = layers.GlobalAveragePooling2D(name="gap")(x)
    x   = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(NUM_CLASSES, name="logits")(x)

    return keras.Model(inputs=inp, outputs=out, name="BCNN_STE")


student_model = build_student()
student_model.summary()


## Step 4 — Knowledge Distillation Training

### Loss function

$$
\mathcal{L} = \alpha \cdot T^2 \cdot D_{KL}\!\left(\sigma\!\left(\frac{z_T}{T}\right) \;\Big\|\; \sigma\!\left(\frac{z_S}{T}\right)\right) + (1-\alpha) \cdot \text{CE}(y, z_S)
$$

where $z_T$ = teacher logits, $z_S$ = student logits, $T$ = temperature,
$\alpha$ = soft-label weight, $\sigma$ = softmax.

The $T^2$ factor compensates for the gradient magnitude reduction caused by
temperature scaling.  We use $T=4$, $\alpha=0.7$.

### Workflow
1. Run teacher on the entire training set once to get soft labels (fast, done once)
2. Train student with combined hard + soft loss using a custom `tf.function` loop


In [ ]:
# ── Pre-compute teacher soft labels at temperature T ─────────────────────────
print("Computing teacher soft labels on training set …")

teacher_logits_train = teacher_model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
teacher_soft_train   = scipy_softmax(teacher_logits_train / KD_TEMPERATURE, axis=1).astype(np.float32)

teacher_logits_val   = teacher_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
teacher_soft_val     = scipy_softmax(teacher_logits_val / KD_TEMPERATURE, axis=1).astype(np.float32)

print(f"Soft label shape : {teacher_soft_train.shape}")
print(f"Soft label range : [{teacher_soft_train.min():.3f}, {teacher_soft_train.max():.3f}]")

# ── Build tf.data pipelines ───────────────────────────────────────────────────
# Dataset yields (image, hard_label, teacher_soft_label) tuples.
# cache() stores data in RAM after the first epoch so subsequent epochs
# never re-read or re-preprocess — critical speedup for multi-epoch GPU training.

def make_dataset(X, y, soft, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y, soft))
    ds = ds.cache()                      # cache raw samples in RAM
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ds_train_kd = make_dataset(X_train, y_train, teacher_soft_train, shuffle=True)
ds_val_kd   = make_dataset(X_val,   y_val,   teacher_soft_val,   shuffle=False)

print(f"Training batches: {len(ds_train_kd)}  |  Val batches: {len(ds_val_kd)}")


In [ ]:
# ── KD training loop ──────────────────────────────────────────────────────────

optimizer = Adam(STUDENT_LR)
ce_loss   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
kl_loss   = tf.keras.losses.KLDivergence()

# Metric trackers
train_loss_m = tf.keras.metrics.Mean(name="train_loss")
train_acc_m  = tf.keras.metrics.SparseCategoricalAccuracy(name="train_acc")
val_loss_m   = tf.keras.metrics.Mean(name="val_loss")
val_acc_m    = tf.keras.metrics.SparseCategoricalAccuracy(name="val_acc")


@tf.function
def train_step(x, y_hard, y_soft):
    with tf.GradientTape() as tape:
        logits     = student_model(x, training=True)
        hard       = ce_loss(y_hard, logits)
        # Temperature-scaled KD loss
        s_soft     = tf.nn.softmax(tf.cast(logits, tf.float32) / KD_TEMPERATURE)
        soft       = kl_loss(y_soft, s_soft) * (KD_TEMPERATURE ** 2)
        # L2 regularisation (from kernel_regularizer in BinaryConv2D)
        reg        = tf.add_n(student_model.losses) if student_model.losses else 0.0
        total_loss = KD_ALPHA * soft + (1.0 - KD_ALPHA) * hard + reg
    grads = tape.gradient(total_loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))
    train_loss_m.update_state(total_loss)
    train_acc_m.update_state(y_hard, logits)


@tf.function
def val_step(x, y_hard, y_soft):
    logits = student_model(x, training=False)
    hard   = ce_loss(y_hard, logits)
    s_soft = tf.nn.softmax(tf.cast(logits, tf.float32) / KD_TEMPERATURE)
    soft   = kl_loss(y_soft, s_soft) * (KD_TEMPERATURE ** 2)
    total  = KD_ALPHA * soft + (1.0 - KD_ALPHA) * hard
    val_loss_m.update_state(total)
    val_acc_m.update_state(y_hard, logits)


# ── Training loop with early stopping ─────────────────────────────────────────
best_val_loss = float("inf")
patience_ctr  = 0
PATIENCE      = 12
best_weights  = None
history_kd    = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"Training student with KD  (T={KD_TEMPERATURE}, α={KD_ALPHA}, "
      f"patience={PATIENCE}, max_epochs={STUDENT_EPOCHS}) …\n")

for epoch in range(1, STUDENT_EPOCHS + 1):
    train_loss_m.reset_state(); train_acc_m.reset_state()
    val_loss_m.reset_state();   val_acc_m.reset_state()

    for x_b, y_b, s_b in ds_train_kd:
        train_step(x_b, y_b, s_b)
    for x_b, y_b, s_b in ds_val_kd:
        val_step(x_b, y_b, s_b)

    tl = float(train_loss_m.result())
    ta = float(train_acc_m.result())
    vl = float(val_loss_m.result())
    va = float(val_acc_m.result())

    history_kd["train_loss"].append(tl)
    history_kd["train_acc"].append(ta)
    history_kd["val_loss"].append(vl)
    history_kd["val_acc"].append(va)

    improved = vl < best_val_loss
    if improved:
        best_val_loss = vl
        patience_ctr  = 0
        best_weights  = student_model.get_weights()
        flag = "✓"
    else:
        patience_ctr += 1
        flag = ""

    print(f"Epoch {epoch:3d}/{STUDENT_EPOCHS}  "
          f"loss={tl:.4f}  acc={ta:.4f}  "
          f"val_loss={vl:.4f}  val_acc={va:.4f}  {flag}")

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
        break

# Restore best weights
student_model.set_weights(best_weights)
student_model.save(str(MODEL_DIR / "student_ste_kd.keras"))
print("\nBest student weights restored and saved.")


In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history_kd["train_loss"], label="train")
ax1.plot(history_kd["val_loss"],   label="val")
ax1.set_title("KD Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history_kd["train_acc"], label="train")
ax2.plot(history_kd["val_acc"],   label="val")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Student (STE + KD) training curves", fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / "student_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Evaluate student (float, before binarization) ────────────────────────────
y_pred_student = np.argmax(student_model.predict(X_test, verbose=0), axis=1)
student_float_acc = accuracy_score(y_test, y_pred_student)

print(f"\n{'─'*55}")
print(f"  Teacher  (MobileNetV2, float32)  : {teacher_acc*100:.2f}%")
print(f"  Student  (STE-BCNN, float32)     : {student_float_acc*100:.2f}%")
print(f"{'─'*55}")
print(classification_report(y_test, y_pred_student,
                             target_names=["Rock", "Not-Rock"]))


## Step 5 — Weight Binarization & BatchNorm Folding

### Weight binarization
`W_bin = sign(W_float)` with `sign(0) = +1`.

Because we used STE during training, weights already cluster near ±1 — the
sign operation causes almost no information loss (unlike post-hoc binarization
of a float model).

### BatchNorm folding → popcount thresholds

After a binary conv layer the output is:

$$y_f = \gamma_f \cdot \frac{\text{dot}_f - \mu_f}{\sigma_f} + \beta_f$$

The binary activation is `sign(y_f)`, which equals `+1` iff $y_f > 0$, i.e.:

$$\text{dot}_f \geq \underbrace{\mu_f - \frac{\beta_f \sigma_f}{\gamma_f}}_{\theta^{\text{dot}}_f} \quad (\gamma_f > 0, \text{ else flip})$$

Since `dot = 2·popcount − N` (where `N = kH·kW·C_in`):

$$\theta^{\text{pc}}_f = \left\lceil \frac{N + \theta^{\text{dot}}_f}{2} \right\rceil$$

This single integer per filter is what the RTL compares against the popcount.


In [ ]:
def binarize_conv_weights(model: keras.Model, layer_name: str):
    """Extract and binarize conv weights from a BinaryConv2D layer."""
    layer   = model.get_layer(layer_name)
    W_float = layer.kernel.numpy()        # (kH, kW, C_in, C_out)
    W_sign  = np.sign(W_float)
    W_sign[W_sign == 0] = 1
    return W_float, W_sign.astype(np.int8)


def fold_bn_thresholds(W_bin: np.ndarray, bn_layer) -> dict:
    """
    Fold BatchNorm parameters into per-filter popcount thresholds.

    Returns a dict with keys:
      thresholds_pc  : (C_out,) int32  — popcount threshold per filter
      thresholds_dot : (C_out,) float  — threshold in dot-product space (for α-correction)
      flip_flags     : (C_out,) bool   — True when gamma < 0 (flip comparison)
      N              : int             — kernel element count (kH*kW*C_in)
    """
    gamma = bn_layer.gamma.numpy()
    beta  = bn_layer.beta.numpy()
    mu    = bn_layer.moving_mean.numpy()
    var   = bn_layer.moving_variance.numpy()
    eps   = bn_layer.epsilon
    sigma = np.sqrt(var + eps)

    kH, kW, C_in, C_out = W_bin.shape
    N = kH * kW * C_in

    # Float dot-product threshold: dot >= θ_dot  →  sign(BN(dot)) = +1 (when γ > 0)
    theta_dot = mu - (beta * sigma / gamma)

    # Convert to popcount space:  dot = 2*pc - N  →  pc = (N + dot) / 2
    theta_pc = np.clip(
        np.ceil((N + theta_dot) / 2).astype(np.int32), 0, N
    )

    return {
        "thresholds_pc" : theta_pc,
        "thresholds_dot": theta_dot,
        "flip_flags"    : (gamma < 0),
        "N"             : N,
    }


# ── Extract weights & fold BN ─────────────────────────────────────────────────
W1_float, W1_bin = binarize_conv_weights(student_model, "bin_conv1")
W2_float, W2_bin = binarize_conv_weights(student_model, "bin_conv2")

L1_thresh = fold_bn_thresholds(W1_bin, student_model.get_layer("bn1"))
L2_thresh = fold_bn_thresholds(W2_bin, student_model.get_layer("bn2"))

# Dense layer weights (kept in float / fixed-point for RTL)
W_dense = student_model.get_layer("logits").get_weights()[0]  # (16, 2)
b_dense = student_model.get_layer("logits").get_weights()[1]  # (2,)

print("Layer 1 thresholds (popcount):", L1_thresh["thresholds_pc"].tolist())
print("Layer 1 flip flags           :", L1_thresh["flip_flags"].tolist())
print("Layer 2 thresholds (popcount):", L2_thresh["thresholds_pc"].tolist())
print("Layer 2 flip flags           :", L2_thresh["flip_flags"].tolist())

# ── Weight histogram: verify STE pushed weights to ±1 ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, W, title in zip(axes, [W1_float.ravel(), W2_float.ravel()],
                         ["Conv1 weight distribution", "Conv2 weight distribution"]):
    ax.hist(W, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(-1, color="red",   ls="--", lw=1.5, label="±1 target")
    ax.axvline(+1, color="red",   ls="--", lw=1.5)
    ax.axvline( 0, color="black", ls=":",  lw=1.0, label="0")
    ax.set_title(title); ax.set_xlabel("Weight value"); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("STE-trained weights cluster near ±1", fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR / "weight_histogram.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 6 — Hardware File Generation

Generates:
- **Binary weight .mif files** — Quartus ROM initialisation for conv1 / conv2
- **Threshold .hex files** — popcount thresholds for each layer
- **Dense layer .hex files** — Q8.8 fixed-point weights & biases
- **Test image .hex files** — for `$readmemh` testbench stimulus
- **rtl_manifest.json** — single source of truth for all RTL parameters


In [ ]:
# ── Fixed-point helpers ───────────────────────────────────────────────────────

def to_q8_8(x: np.ndarray) -> np.ndarray:
    """Quantise float array to Q8.8 fixed-point (int16, scale=256)."""
    scaled  = np.round(x * FP_SCALE).astype(np.int32)
    clipped = np.clip(scaled, *FP_CLIP).astype(np.int16)
    return clipped


def signed_to_twos_complement(vals: np.ndarray, bits: int) -> np.ndarray:
    """Convert signed integers to their two's complement representation."""
    mask = (1 << bits) - 1
    return np.array([int(v) & mask for v in vals], dtype=np.uint32)


def write_hex(path: Path, data: np.ndarray, bits: int = 16):
    """Write array to a $readmemh-compatible hex file."""
    hex_chars = math.ceil(bits / 4)
    with open(path, "w") as f:
        for v in data.flatten():
            f.write(f"{int(v) & ((1 << bits) - 1):0{hex_chars}X}\n")


# ── Binary weight flat arrays (1 bit per element, packed as hex) ──────────────

def weights_to_bits(W_bin: np.ndarray) -> np.ndarray:
    """Convert ±1 weight array to 0/1 bits, flattened in FPGA-friendly order."""
    return ((W_bin + 1) // 2).astype(np.uint8).ravel()


W1_flat = weights_to_bits(W1_bin)   # (kH*kW*C_in*C_out,)
W2_flat = weights_to_bits(W2_bin)


def write_mif(path: Path, bits: np.ndarray, width: int = 1):
    """Write a Quartus .mif file for a 1-bit-wide ROM."""
    depth = len(bits)
    with open(path, "w") as f:
        f.write(f"DEPTH = {depth};\n")
        f.write(f"WIDTH = {width};\n")
        f.write("ADDRESS_RADIX = UNS;\n")
        f.write("DATA_RADIX = BIN;\n")
        f.write("CONTENT BEGIN\n")
        for addr, val in enumerate(bits):
            f.write(f"  {addr} : {val:0{width}b};\n")
        f.write("END;\n")


write_mif(MEM_DIR / "conv1_weights.mif", W1_flat)
write_mif(MEM_DIR / "conv2_weights.mif", W2_flat)
print(f"Conv1 weight bits : {len(W1_flat)}")
print(f"Conv2 weight bits : {len(W2_flat)}")

# ── Threshold hex files ───────────────────────────────────────────────────────

COUNT_W_L1 = math.ceil(math.log2(L1_thresh["N"] + 2))
COUNT_W_L2 = math.ceil(math.log2(L2_thresh["N"] + 2))

L1_th_tc = signed_to_twos_complement(L1_thresh["thresholds_pc"], bits=COUNT_W_L1)
L2_th_tc = signed_to_twos_complement(L2_thresh["thresholds_pc"], bits=COUNT_W_L2)

write_hex(HEX_DIR / "conv1_thresh.hex", L1_th_tc, bits=COUNT_W_L1)
write_hex(HEX_DIR / "conv2_thresh.hex", L2_th_tc, bits=COUNT_W_L2)

# ── Dense layer hex files (Q8.8) ──────────────────────────────────────────────

Wq = to_q8_8(W_dense)   # (16, 2)
bq = to_q8_8(b_dense)   # (2,)

write_hex(HEX_DIR / "dense_w0.hex", Wq[:, 0])
write_hex(HEX_DIR / "dense_w1.hex", Wq[:, 1])
write_hex(HEX_DIR / "dense_b.hex",  bq)

print("Dense layer exported (Q8.8)")

# ── RTL manifest ──────────────────────────────────────────────────────────────

manifest = {
    "image"  : {"H": IMG_H, "W": IMG_W, "C": IMG_C, "pixel_bin_thr": PIXEL_BIN_THR},
    "layer1" : {
        "filters": CONV1_FILTERS, "kernel": KERNEL_SIZE,
        "N": L1_thresh["N"], "count_width": COUNT_W_L1,
        "thresholds": L1_thresh["thresholds_pc"].tolist(),
        "flip_flags": L1_thresh["flip_flags"].tolist(),
        "weight_bits": len(W1_flat),
    },
    "layer2" : {
        "filters": CONV2_FILTERS, "kernel": KERNEL_SIZE,
        "N": L2_thresh["N"], "count_width": COUNT_W_L2,
        "thresholds": L2_thresh["thresholds_pc"].tolist(),
        "flip_flags": L2_thresh["flip_flags"].tolist(),
        "weight_bits": len(W2_flat),
    },
    "dense"  : {"in_features": CONV2_FILTERS, "out_features": NUM_CLASSES,
                "fixed_point": fp_spec},
}

manifest_path = OUTPUT_DIR / "rtl_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\nRTL manifest → {manifest_path}")


In [ ]:
# ── Test-image hex files (for $readmemh testbench stimulus) ───────────────────

N_TEST_IMAGES = 10   # number of testbench stimulus images

# Grab a balanced selection (5 Rock, 5 Not-Rock)
rock_idx    = np.where(y_test == 0)[0][:5]
notrock_idx = np.where(y_test == 1)[0][:5]
tb_idx      = np.concatenate([rock_idx, notrock_idx])

X_tb = X_test[tb_idx]   # (10, 32, 32, 1)
y_tb = y_test[tb_idx]

def image_to_hex(path: Path, img_float: np.ndarray, threshold: int = PIXEL_BIN_THR):
    """
    Write a 32×32 binary image as a $readmemh hex file.
    Each pixel becomes 1 byte: 0x01 (>=threshold) or 0x00 (<threshold).
    """
    px_u8  = (img_float * 255).astype(np.uint8)
    px_bin = (px_u8 >= threshold).astype(np.uint8)
    with open(path, "w") as f:
        for val in px_bin.ravel():
            f.write(f"{val:02X}\n")

for i, img in enumerate(X_tb[..., 0]):
    image_to_hex(HEX_DIR / f"test_img_{i:02d}.hex", img)

print(f"Exported {N_TEST_IMAGES} testbench images → {HEX_DIR}/test_img_*.hex")


## Step 7 — Software XNOR Inference (RTL Golden Reference)

This cell implements the exact same computation the RTL performs, in NumPy.
It serves as the **golden reference**: if the FPGA output matches this,
the RTL is correct.

### Speed improvement

The original implementation had three nested Python `for` loops
(`H_out × W_out × C_out`), which made it extremely slow (~minutes).

The new implementation uses `numpy.lib.stride_tricks.sliding_window_view`
to extract all windows at once, then performs the XNOR-popcount as a single
batched matrix operation.  For a 32×32 image with 16 filters this reduces
Python loop iterations from 16,384 to 0 — the entire layer runs in one NumPy call.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view


def binarize_pixels(img_float: np.ndarray, threshold: int = PIXEL_BIN_THR) -> np.ndarray:
    """
    img_float : (H, W) float32 in [0, 1]
    Returns   : (H, W) int8 in {-1, +1}
    """
    px_u8 = (img_float * 255).astype(np.uint8)
    return np.where(px_u8 >= threshold, np.int8(1), np.int8(-1))


def xnor_conv2d(
    act_bin   : np.ndarray,    # (H, W, C_in) int8 {-1, +1}
    W_bin     : np.ndarray,    # (kH, kW, C_in, C_out) int8 {-1, +1}
    thresholds: np.ndarray,    # (C_out,) int32
    flip_flags: np.ndarray,    # (C_out,) bool
    padding   : str = "same",
) -> np.ndarray:
    """
    Vectorized binary convolution via XNOR-popcount + folded BN threshold.

    Returns (H_out, W_out, C_out) int8 in {-1, +1}.

    Padding uses -1 (binary zero — contributes 0 to popcount, mirroring RTL).
    """
    kH, kW, C_in, C_out = W_bin.shape
    pad_h, pad_w = (kH - 1) // 2, (kW - 1) // 2

    if padding == "same":
        act_bin = np.pad(
            act_bin, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)),
            mode="constant", constant_values=-1,
        )

    # sliding_window_view: (H_out, W_out, 1, kH, kW, C_in) — squeeze axis 2
    windows = sliding_window_view(act_bin, (kH, kW, C_in))[:, :, 0, :, :, :]
    # windows shape: (H_out, W_out, kH, kW, C_in)

    # XNOR agreement map: {0, 1}
    # windows[..., newaxis] : (H_out, W_out, kH, kW, C_in, 1)
    # W_bin[newaxis, newaxis]: (1, 1, kH, kW, C_in, C_out)
    xnor = (windows[..., np.newaxis].astype(np.int32)
            * W_bin[np.newaxis, np.newaxis].astype(np.int32) + 1) // 2
    # shape: (H_out, W_out, kH, kW, C_in, C_out)

    popcounts = xnor.sum(axis=(2, 3, 4))   # (H_out, W_out, C_out)

    # Threshold comparison — broadcast over spatial dims
    above = popcounts >= thresholds[np.newaxis, np.newaxis, :]   # bool

    # Flip comparison for filters where BN gamma < 0
    result = np.where(flip_flags[np.newaxis, np.newaxis, :], ~above, above)

    return np.where(result, np.int8(1), np.int8(-1))


def gap_binary(act_bin: np.ndarray) -> np.ndarray:
    """
    Global average pool of binary activations.
    Returns (C,) float32 — mean of ±1 values, i.e. fraction of agreements.
    """
    return act_bin.mean(axis=(0, 1)).astype(np.float32)


def bcnn_infer(
    img_float: np.ndarray,   # (H, W) float32 [0, 1]
    W1_bin, L1_info,
    W2_bin, L2_info,
    W_dense, b_dense,
    Wq, bq,
    pixel_thr: int = PIXEL_BIN_THR,
) -> tuple[int, np.ndarray]:
    """
    Full binarized inference pipeline (mirrors RTL exactly).
    Returns (predicted_class, q8_8_logits).
    """
    # 1. Input binarization
    x = binarize_pixels(img_float, pixel_thr)[..., np.newaxis]   # (H, W, 1)

    # 2. Layer 1 — binary conv
    x = xnor_conv2d(x, W1_bin,
                    L1_info["thresholds_pc"], L1_info["flip_flags"])   # (H, W, 8)

    # 3. Layer 2 — binary conv
    x = xnor_conv2d(x, W2_bin,
                    L2_info["thresholds_pc"], L2_info["flip_flags"])   # (H, W, 16)

    # 4. Global average pool → float feature vector
    gap = gap_binary(x)   # (16,) float32

    # 5. Dense layer in Q8.8 fixed-point (bit-accurate to RTL)
    gap_q    = to_q8_8(gap)
    logits_q = np.zeros(2, dtype=np.int32)
    for i in range(2):
        logits_q[i] = (np.int32(gap_q) * np.int32(Wq[:, i])).sum() + np.int32(bq[i])

    return int(np.argmax(logits_q)), logits_q


## Step 8 — XNORNet α Correction + Dense Head Refit

Even with STE training, two small corrections further close the
training-inference gap:

**α correction (weight scaling):**
XNORNet shows that the binary dot product should be scaled by
$\alpha_f = \text{mean}|W_f|$ to restore the magnitude of the float
dot product before comparing with BN thresholds.  With STE weights
already near ±1 the correction is small, but we still apply it.

**Dense head refit:**
Collect the actual binary GAP vectors produced by the binarized network
on the full training set, then fit a fresh logistic regression head on
those exact features.  This ensures the dense weights are calibrated for
the real ±1 inputs they'll see at inference — no RTL change needed.


In [ ]:
# ── α-corrected thresholds ────────────────────────────────────────────────────
alpha1 = np.abs(W1_float).mean(axis=(0, 1, 2))   # (C_out=8,)
alpha2 = np.abs(W2_float).mean(axis=(0, 1, 2))   # (C_out=16,)

print(f"Layer 1 α (should be ≈1.0 with STE): {alpha1.round(3)}")
print(f"Layer 2 α (should be ≈1.0 with STE): {alpha2.round(3)}")

def alpha_correct_thresholds(thresh_info: dict, alpha: np.ndarray) -> dict:
    """Apply XNORNet weight-scaling correction to popcount thresholds."""
    N        = thresh_info["N"]
    th_dot_c = thresh_info["thresholds_dot"] / alpha          # corrected dot threshold
    th_pc_c  = np.clip(
        np.ceil((N + th_dot_c) / 2).astype(np.int32), 0, N
    )
    return {**thresh_info, "thresholds_pc": th_pc_c, "thresholds_dot": th_dot_c}

L1_thresh_xnor = alpha_correct_thresholds(L1_thresh, alpha1)
L2_thresh_xnor = alpha_correct_thresholds(L2_thresh, alpha2)

print("\nLayer 1  original → α-corrected thresholds:")
print(" ", L1_thresh["thresholds_pc"].tolist(), "→", L1_thresh_xnor["thresholds_pc"].tolist())
print("Layer 2  original → α-corrected thresholds:")
print(" ", L2_thresh["thresholds_pc"].tolist(), "→", L2_thresh_xnor["thresholds_pc"].tolist())


In [ ]:
# ── Collect binary GAP features from training set ────────────────────────────
# Uses ThreadPoolExecutor so NumPy XNOR ops run in parallel across CPU cores.
# NumPy releases the GIL for vectorized ops, so threads give real parallelism.

def collect_gap_features(imgs_float, W1, L1_info, W2, L2_info, desc="GAP features"):
    """Parallel binary GAP feature extraction (one thread per image batch)."""
    def _process_one(img):
        x = binarize_pixels(img)[..., np.newaxis]
        x = xnor_conv2d(x, W1, L1_info["thresholds_pc"], L1_info["flip_flags"])
        x = xnor_conv2d(x, W2, L2_info["thresholds_pc"], L2_info["flip_flags"])
        return gap_binary(x)

    n_workers = min(os.cpu_count() or 4, 8)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        results = list(tqdm(
            pool.map(_process_one, imgs_float),
            total=len(imgs_float), desc=desc, unit="img",
        ))
    return np.array(results)

print(f"Collecting binary GAP features (parallel, {min(os.cpu_count() or 4, 8)} workers) …\n")
gap_train = collect_gap_features(
    X_train[..., 0], W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
    desc="GAP features (train)",
)
gap_val = collect_gap_features(
    X_val[..., 0], W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
    desc="GAP features (val)  ",
)

print(f"\nGAP feature matrix : {gap_train.shape}")
print(f"Value range        : [{gap_train.min():.3f}, {gap_train.max():.3f}]")


In [ ]:
# ── Fit logistic regression head on binary features ───────────────────────────
lr_head = LogisticRegression(
    class_weight="balanced",
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=2000,
    C=1.0,
)
lr_head.fit(gap_train, y_train)

print(f"Head refit — train acc : {accuracy_score(y_train, lr_head.predict(gap_train))*100:.1f}%")
print(f"Head refit — val acc   : {accuracy_score(y_val,   lr_head.predict(gap_val))*100:.1f}%")
print(f"coef_ shape            : {lr_head.coef_.shape}")

# ── Extract weights, handling sklearn's binary vs. multinomial coef_ shape ───
# sklearn returns coef_ shape (1, n_features) for binary problems (even with
# multi_class='multinomial' on some solver+version combos) and (2, n_features)
# for explicit 2-class multinomial.  bcnn_infer always expects (n_features, 2)
# and (2,), so we normalise here.

if lr_head.coef_.shape[0] == 1:
    # Binary OVR layout: one weight vector w; decision = w·x + b > 0 → class 1.
    # Expand to two-class logits: class-0 logit = -(w·x+b), class-1 = +(w·x+b).
    w = lr_head.coef_[0].astype(np.float32)          # (n_features,)
    b = float(lr_head.intercept_[0])
    W_dense_refit = np.stack([-w, w], axis=1)         # (n_features, 2)
    b_dense_refit = np.array([-b, b], dtype=np.float32)  # (2,)
else:
    # Multinomial layout: coef_ is (n_classes, n_features)
    W_dense_refit = lr_head.coef_.T.astype(np.float32)   # (n_features, 2)
    b_dense_refit = lr_head.intercept_.astype(np.float32) # (2,)

print(f"W_dense_refit shape    : {W_dense_refit.shape}   (expect ({CONV2_FILTERS}, {NUM_CLASSES}))")
print(f"b_dense_refit shape    : {b_dense_refit.shape}   (expect ({NUM_CLASSES},))")

Wq_refit = to_q8_8(W_dense_refit)
bq_refit = to_q8_8(b_dense_refit)


## Evaluation — Full Test-Set Accuracy Comparison

In [ ]:
# ── Run binarized inference (α-corrected + refit head) on test set ────────────
# Parallelised across CPU cores — NumPy releases GIL so threads overlap.
print("Running binarized inference on test set (parallel) …\n")

def _infer_one_eval(img):
    pred, _ = bcnn_infer(
        img, W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
        W_dense_refit, b_dense_refit, Wq_refit, bq_refit,
    )
    return pred

n_workers = min(os.cpu_count() or 4, 8)
with ThreadPoolExecutor(max_workers=n_workers) as pool:
    y_pred_bin = np.array(list(tqdm(
        pool.map(_infer_one_eval, X_test[..., 0]),
        total=len(X_test), desc="Binary inference", unit="img",
    )))

bin_acc = accuracy_score(y_test, y_pred_bin)

print(f"\n{'═'*60}")
print(f"  ACCURACY SUMMARY")
print(f"{'─'*60}")
print(f"  Teacher  (MobileNetV2, float)    : {teacher_acc*100:.2f}%")
print(f"  Student  (STE-BCNN, float)       : {student_float_acc*100:.2f}%")
print(f"  Student  (STE-BCNN, binarized)   : {bin_acc*100:.2f}%  ← RTL golden reference")
print(f"  Float → binary accuracy drop     : {(student_float_acc - bin_acc)*100:.2f} pp")
print(f"{'═'*60}")
print(classification_report(y_test, y_pred_bin,
                             target_names=["Rock", "Not-Rock"], zero_division=0))

# ── Three-way confusion matrix ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, preds, title, cmap in zip(
    axes,
    [y_pred_teacher, y_pred_student, y_pred_bin],
    [f"Teacher MobileNetV2 ({teacher_acc*100:.1f}%)",
     f"Student float32 ({student_float_acc*100:.1f}%)",
     f"Student binarized ({bin_acc*100:.1f}%)"],
    ["Purples", "Blues", "Greens"],
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, preds),
        display_labels=["Rock", "Not-Rock"],
    ).plot(ax=ax, cmap=cmap)
    ax.set_title(title)

plt.tight_layout()
plt.savefig(PLOT_DIR / "confusion_three_way.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Golden reference for testbench ─────────────────────────────────────────────
print("Golden reference outputs for RTL testbench:\n")
print(f"{'Img':>4}  {'GT':>10}  {'Pred':>10}  {'OK?':>5}  {'Logit[0]':>10}  {'Logit[1]':>10}")
print("─" * 58)

tb_results = []
for i in range(N_TEST_IMAGES):
    img   = X_tb[i, ..., 0]
    label = int(y_tb[i])
    pred, logits = bcnn_infer(
        img, W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
        W_dense_refit, b_dense_refit, Wq_refit, bq_refit,
    )
    ok = pred == label
    tb_results.append({"img": i, "label": label, "pred": pred, "ok": ok})
    sym = "✓" if ok else "✗"
    gt  = "Rock    " if label == 0 else "Not-Rock"
    pr  = "Rock    " if pred  == 0 else "Not-Rock"
    print(f"  {i:02d}  {gt:>10}  {pr:>10}  {sym:>5}  {logits[0]:>10}  {logits[1]:>10}")

tb_acc = sum(r["ok"] for r in tb_results) / N_TEST_IMAGES
print(f"\nTestbench accuracy: {tb_acc*100:.0f}%  ({sum(r['ok'] for r in tb_results)}/{N_TEST_IMAGES})")

golden_path = HEX_DIR / "golden_outputs.txt"
with open(golden_path, "w") as f:
    f.write("// Golden expected outputs for RTL testbench\n")
    f.write("// Format: img_index  gt_label  expected_pred\n")
    for r in tb_results:
        f.write(f"{r['img']:02d}  {r['label']}  {r['pred']}\n")
print(f"\nGolden reference → {golden_path}")

# ── Update RTL manifest with corrected thresholds and refit weights ────────────
with open(manifest_path) as f:
    manifest = json.load(f)

manifest["layer1"]["thresholds"] = L1_thresh_xnor["thresholds_pc"].tolist()
manifest["layer1"]["alpha"]      = alpha1.round(6).tolist()
manifest["layer2"]["thresholds"] = L2_thresh_xnor["thresholds_pc"].tolist()
manifest["layer2"]["alpha"]      = alpha2.round(6).tolist()

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# Overwrite hex files with α-corrected thresholds and refit dense weights
L1_th_xnor_tc = signed_to_twos_complement(L1_thresh_xnor["thresholds_pc"], bits=COUNT_W_L1)
L2_th_xnor_tc = signed_to_twos_complement(L2_thresh_xnor["thresholds_pc"], bits=COUNT_W_L2)
write_hex(HEX_DIR / "conv1_thresh.hex", L1_th_xnor_tc, bits=COUNT_W_L1)
write_hex(HEX_DIR / "conv2_thresh.hex", L2_th_xnor_tc, bits=COUNT_W_L2)
write_hex(HEX_DIR / "dense_w0.hex", Wq_refit[:, 0])
write_hex(HEX_DIR / "dense_w1.hex", Wq_refit[:, 1])
write_hex(HEX_DIR / "dense_b.hex",  bq_refit)
print("RTL hex files updated with final corrected values.")


## Step 9 — Inference Timing (CPU Baseline for FPGA Comparison)

We time **single-image** inference (no batching) for two implementations:
- **Float32 Keras model** — optimised CPU path
- **Software-binarized BCNN** — pure NumPy, mirrors RTL computation exactly

The software-binarized time is your CPU golden-reference latency.
Your FPGA should be orders of magnitude faster; fill in the Quartus
timing report value below to compute the speedup.

> **Note:** GPU is intentionally not used for the binarized timing because
> the goal is to compare against FPGA — a CPU reference is more meaningful.


In [ ]:
# ── Step 9: Inference Timing — CPU only (FPGA baseline comparison) ───────────
# Both measurements are deliberately pinned to CPU.
#   • float32 Keras  — tf.device('/CPU:0') forces ops onto CPU
#   • SW-binarized   — pure NumPy; already CPU
# This gives a fair apples-to-apples latency that FPGA will be compared against.

N_TIMING_RUNS = 200
N_WARMUP      = 5

img_timing = X_test[0, ..., 0]        # (32, 32) float32
img_batch  = tf.constant(X_test[0:1]) # (1, 32, 32, 1) as tf.Tensor

# ── Float32 Keras — single image, pinned to CPU ───────────────────────────────
print("Timing float32 Keras model on CPU …")

# Warm-up (also traces the tf.function graph on CPU the first time)
for _ in range(N_WARMUP):
    with tf.device('/CPU:0'):
        _ = student_model(img_batch, training=False)

t_float = []
for _ in range(N_TIMING_RUNS):
    t0 = time.perf_counter()
    with tf.device('/CPU:0'):
        _ = student_model(img_batch, training=False)
    t_float.append((time.perf_counter() - t0) * 1000)

# ── Software-binarized BCNN — pure NumPy, always CPU ─────────────────────────
print("Timing software-binarized BCNN …")
for _ in range(N_WARMUP):
    bcnn_infer(img_timing, W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
               W_dense_refit, b_dense_refit, Wq_refit, bq_refit)

t_bin = []
for _ in range(N_TIMING_RUNS):
    t0 = time.perf_counter()
    bcnn_infer(img_timing, W1_bin, L1_thresh_xnor, W2_bin, L2_thresh_xnor,
               W_dense_refit, b_dense_refit, Wq_refit, bq_refit)
    t_bin.append((time.perf_counter() - t0) * 1000)

t_float_mean, t_float_std = np.mean(t_float), np.std(t_float)
t_bin_mean,   t_bin_std   = np.mean(t_bin),   np.std(t_bin)

print(f"\n{'═'*60}")
print(f"  SINGLE-IMAGE INFERENCE LATENCY  (n={N_TIMING_RUNS}, 32×32 grayscale, CPU)")
print(f"{'─'*60}")
print(f"  Float32 Keras (CPU)  : {t_float_mean:8.3f} ms  ± {t_float_std:.3f} ms")
print(f"  SW-binarized  (CPU)  : {t_bin_mean:8.3f} ms  ± {t_bin_std:.3f} ms")
print(f"  SW speedup           : {t_float_mean/t_bin_mean:.2f}× (binarized vs float)")
print(f"{'─'*60}")
print(f"  FPGA target latency  : ________ ms   ← fill from Quartus timing report")
print(f"  FPGA speedup vs SW   : ________ ×    (= {t_bin_mean:.3f} / FPGA_ms)")
print(f"{'═'*60}")

# ── Save timing record ────────────────────────────────────────────────────────
timing = {
    "n_runs"           : N_TIMING_RUNS,
    "device"           : "CPU",
    "float_keras_ms"   : {"mean": round(t_float_mean, 4), "std": round(t_float_std, 4)},
    "sw_binary_ms"     : {"mean": round(t_bin_mean,   4), "std": round(t_bin_std,   4)},
    "sw_speedup"       : round(t_float_mean / t_bin_mean, 3),
    "fpga_target_ms"   : None,
    "fpga_speedup_vs_sw": None,
}
with open(OUTPUT_DIR / "inference_timing.json", "w") as f:
    json.dump(timing, f, indent=2)

# ── Distribution plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, runs, label, color in zip(
    axes,
    [t_float, t_bin],
    [f"Float32 Keras CPU\n({t_float_mean:.2f} ms)", f"SW-binarized BCNN CPU\n({t_bin_mean:.2f} ms)"],
    ["steelblue", "darkorange"],
):
    ax.hist(runs, bins=40, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(np.mean(runs), color="red", ls="--", lw=1.5,
               label=f"mean = {np.mean(runs):.2f} ms")
    ax.set_xlabel("Latency (ms)"); ax.set_ylabel("Count")
    ax.set_title(label); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Single-image inference latency — CPU baseline", fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / "inference_timing.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 10 — Summary & Export

In [ ]:
print("\n" + "═"*65)
print("  BCNN Phase A — COMPLETE SUMMARY")
print("═"*65)

print(f"\n  ACCURACY")
print(f"  {'Teacher (MobileNetV2, float)':45s}: {teacher_acc*100:.2f}%")
print(f"  {'Student (STE-BCNN, float)':45s}: {student_float_acc*100:.2f}%")
print(f"  {'Student (STE-BCNN, binarized)  ← RTL ref':45s}: {bin_acc*100:.2f}%")
print(f"  {'Float → binary accuracy drop':45s}: {(student_float_acc - bin_acc)*100:.2f} pp")

print(f"\n  HARDWARE PARAMETERS")
print(f"  {'Image':30s}: {IMG_H}×{IMG_W} grayscale")
print(f"  {'Pixel binarization threshold':30s}: {PIXEL_BIN_THR}/255")
print(f"  {'Conv1 filters':30s}: {CONV1_FILTERS}  (N={L1_thresh_xnor['N']}, count_w={COUNT_W_L1})")
print(f"  {'Conv2 filters':30s}: {CONV2_FILTERS}  (N={L2_thresh_xnor['N']}, count_w={COUNT_W_L2})")
print(f"  {'Total weight bits (conv1)':30s}: {len(W1_flat)}")
print(f"  {'Total weight bits (conv2)':30s}: {len(W2_flat)}")

print(f"\n  LAYER 1 THRESHOLDS  (N={L1_thresh_xnor['N']})")
print(f"  {L1_thresh_xnor['thresholds_pc'].tolist()}")
print(f"  Flip flags: {L1_thresh_xnor['flip_flags'].tolist()}")

print(f"\n  LAYER 2 THRESHOLDS  (N={L2_thresh_xnor['N']})")
print(f"  {L2_thresh_xnor['thresholds_pc'].tolist()}")
print(f"  Flip flags: {L2_thresh_xnor['flip_flags'].tolist()}")

print(f"\n  KNOWLEDGE DISTILLATION")
print(f"  {'Temperature':30s}: {KD_TEMPERATURE}")
print(f"  {'Alpha (soft loss weight)':30s}: {KD_ALPHA}")

print(f"\n  OUTPUT FILES")
for fp in sorted(OUTPUT_DIR.rglob("*")):
    if fp.is_file():
        size = fp.stat().st_size
        rel  = fp.relative_to(OUTPUT_DIR)
        print(f"  {str(rel):<45} {size:>8} bytes")

print("\n" + "═"*65)
print("  Hand-off to Phase B (RTL). Use rtl_manifest.json to")
print("  parameterise all Verilog modules.")
print("═"*65)


In [ ]:
import shutil

zip_base = "/content/output"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Created: {zip_base}.zip")

# Colab download (only works inside Google Colab)
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except ImportError:
    print("(Not running in Colab — find the zip at /content/output.zip)")
